# Phase 1: Setup & Environment

In [6]:
#load and validate config from .env 
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # make project root importable

from dotenv import load_dotenv
load_dotenv("../.env")

from config.settings import get_settings

settings = get_settings()

print("✅ Config loaded")
print(f"   App env      : {settings.app_env}")
print(f"   LLM provider : {settings.llm_provider}")
print(f"   Dry run      : {settings.agent_dry_run}")
print(f"   Auto apply   : {settings.agent_auto_apply}")
print(f"   DB URL       : {settings.database_url[:40]}...")

✅ Config loaded
   App env      : development
   LLM provider : ollama
   Dry run      : True
   Auto apply   : False
   DB URL       : sqlite+aiosqlite:///./job_hunt.db...


In [7]:
# Quick checklist — flags anything missing
checks = {
    "DATABASE_URL set"        : bool(settings.database_url),
    "REDIS_URL set"           : bool(settings.redis_url),
    "LLM_PROVIDER set"         : settings.llm_provider in ("ollama", "openai", "gemini", "qwen"),
    "API key (if needed)"      : settings.llm_provider in ("ollama",) or (
                                 (settings.llm_provider == "gemini" and bool(os.getenv("GOOGLE_API_KEY"))) or
                                 (settings.llm_provider == "qwen"   and bool(os.getenv("QWEN_API_KEY")))   or
                                 (settings.llm_provider == "openai" and bool(settings.openai_api_key))),  
    "LinkedIn email set"      : bool(settings.linkedin_email),
    "Gmail sender set"        : bool(settings.gmail_sender_email),
}

for label, ok in checks.items():
    icon = "✅" if ok else "⚠️ "
    print(f"  {icon}  {label}")

  ✅  DATABASE_URL set
  ✅  REDIS_URL set
  ✅  LLM_PROVIDER set
  ✅  API key (if needed)
  ✅  LinkedIn email set
  ⚠️   Gmail sender set


In [8]:
#database connectivity test
import sqlite3
import os

# Create the database file in your project root
DB_PATH = os.path.abspath("../job_hunt.db")

def test_sqlite():
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute("SELECT sqlite_version()")
        version = cursor.fetchone()[0]
        conn.close()
        return version
    except Exception as e:
        return f"ERROR: {e}"

result = test_sqlite()

if "ERROR" in str(result):
    print(f"❌ SQLite: {result}")
else:
    print(f"✅ SQLite connected")
    print(f"   Version  : {result}")
    print(f"   DB file  : {DB_PATH}")

✅ SQLite connected
   Version  : 3.40.1
   DB file  : c:\AI_ML\Zenith_projects\job_hunt.db


In [9]:
#test LLM connectivity
async def test_llm():
    prompt = "Reply with exactly: LLM connection successful."

    if settings.llm_provider == "ollama":
        import ollama
        try:
            response = ollama.chat(model=settings.ollama_model,
                                   messages=[{"role": "user", "content": prompt}])
            return response["message"]["content"]
        except Exception as e:
            return f"ERROR: {e}"

    elif settings.llm_provider == "openai":
        from openai import AsyncOpenAI
        try:
            client = AsyncOpenAI(api_key=settings.openai_api_key)
            response = await client.chat.completions.create(
                model=settings.openai_model,
                messages=[{"role": "user", "content": prompt}], max_tokens=30)
            return response.choices[0].message.content
        except Exception as e:
            return f"ERROR: {e}"

    # ── NEW: Google Gemini ──
    elif settings.llm_provider == "gemini":
        import google.generativeai as genai
        try:
            genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
            model = genai.GenerativeModel(os.getenv("GEMINI_MODEL", "gemini-1.5-flash"))
            response = model.generate_content(prompt)
            return response.text
        except Exception as e:
            return f"ERROR: {e}"

    # ── NEW: Qwen (Ollama or API) ──
    elif settings.llm_provider == "qwen":
        qwen_model = os.getenv("QWEN_MODEL", "qwen2.5:7b")
        if ":" in qwen_model:          # local Ollama model e.g. qwen2.5:7b
            import ollama
            try:
                response = ollama.chat(model=qwen_model,
                                       messages=[{"role": "user", "content": prompt}])
                return response["message"]["content"]
            except Exception as e:
                return f"ERROR: {e}"
        else:                          # API model e.g. qwen-plus
            from openai import AsyncOpenAI
            try:
                client = AsyncOpenAI(
                    api_key=os.getenv("QWEN_API_KEY"),
                    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
                )
                response = await client.chat.completions.create(
                    model=qwen_model,
                    messages=[{"role": "user", "content": prompt}], max_tokens=30)
                return response.choices[0].message.content
            except Exception as e:
                return f"ERROR: {e}"
            
            
model_label = {
    "ollama" : settings.ollama_model,
    "openai" : settings.openai_model,
    "gemini" : os.getenv("GEMINI_MODEL", "gemini-1.5-flash"),
    "qwen"   : os.getenv("QWEN_MODEL", "qwen2.5:7b"),
}.get(settings.llm_provider, "unknown")
print(f"✅ LLM ({settings.llm_provider} / {model_label})")            


✅ LLM (ollama / llama3)


In [5]:
#test sellenium and chromedriver
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

def get_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-web-security")
    options.add_argument("--dns-prefetch-disable")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

def test_selenium():
    try:
        driver = get_driver()
        # Test with a data URI instead of a real URL — no network needed
        driver.get("data:text/html,<h1>Selenium OK</h1>")
        title_elem = driver.find_element("tag name", "h1")
        text = title_elem.text
        driver.quit()
        return text
    except Exception as e:
        return f"ERROR: {e}"

result = test_selenium()

if "ERROR" in str(result):
    print(f"❌ Selenium: {result}")
else:
    print(f"✅ Selenium + ChromeDriver working")
    print(f"   Chrome version : 146.0.7680.178")
    print(f"   Test result    : '{result}'")
    print(f"   Ready to scrape LinkedIn and Indeed")

✅ Selenium + ChromeDriver working
   Chrome version : 146.0.7680.178
   Test result    : 'Selenium OK'
   Ready to scrape LinkedIn and Indeed


In [10]:
#test fast API health endpoint
import httpx

async def test_api():
    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get("http://localhost:8000/health", timeout=3)
            return resp.json()
    except Exception as e:
        return f"ERROR: {e}"

result = await test_api()

if "ERROR" in str(result):
    print(f"❌ FastAPI not reachable: {result}")
    print("   → Run in terminal: uvicorn api.main:app --reload")
else:
    print(f"✅ FastAPI running")
    print(f"   {result}")

✅ FastAPI running
   {'status': 'ok', 'env': 'development'}


In [11]:
#PHASE 1 SUMMARY
print("=" * 50)
print("  Phase 1 — Environment Checklist")
print("=" * 50)
print()
print("  Fix any ❌ above before moving to Phase 2.")
print()
print("  Phase 2 will cover:")
print("  → PostgreSQL schema (jobs, applications, resumes, feedback)")
print("  → SQLAlchemy models")
print("  → Alembic migration")
print("  → Seed data for testing")
print()
print("  Next notebook: 02_phase2_database_schema.ipynb")

  Phase 1 — Environment Checklist

  Fix any ❌ above before moving to Phase 2.

  Phase 2 will cover:
  → PostgreSQL schema (jobs, applications, resumes, feedback)
  → SQLAlchemy models
  → Alembic migration
  → Seed data for testing

  Next notebook: 02_phase2_database_schema.ipynb


# PHASE 2 - DATABASE

In [21]:
import sqlite3

DB_PATH = r"c:\AI_ML\Zenith_projects\AI_JobHuntAgent\job_hunt.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("SELECT * FROM feedback LIMIT 1")
rows = cursor.fetchall()
print("Rows:", rows)
print("Count:", len(rows))
conn.close()

Rows: [(1, 1, 2025, 15, 3, 1, 0, 71.2, '["Missing Kubernetes", "No cloud certifications"]', 'Kubernetes, AWS, MLOps', 'Add a cloud deployment project to experience section.', 1, '2026-04-11 11:05:27.321414')]
Count: 1


In [12]:
#create tables
import asyncio
from app.db.base import engine, init_db, Base
from app.db.models import Job, Resume, Application, Feedback

await init_db()

# Confirm tables exist
from sqlalchemy import inspect, text

async def list_tables():
    async with engine.connect() as conn:
        result = await conn.execute(text("SELECT name FROM sqlite_master WHERE type='table'"))
        return [row[0] for row in result.fetchall()]

tables = await list_tables()

print("✅ Tables created:")
for t in tables:
    print(f"   → {t}")

✅ Tables created:
   → jobs
   → feedback
   → resumes
   → applications


In [13]:
#check columns per table
async def show_schema():
    async with engine.connect() as conn:
        for table in ["jobs", "resumes", "applications", "feedback"]:
            result = await conn.execute(text(f"PRAGMA table_info({table})"))
            cols = result.fetchall()
            print(f"\n📋 {table} ({len(cols)} columns)")
            for col in cols:
                pk = " ← PK" if col[5] else ""
                nullable = "" if col[3] else " (nullable)"
                print(f"   {col[1]:<25} {col[2]}{pk}{nullable}")

await show_schema()


📋 jobs (17 columns)
   id                        INTEGER ← PK
   source                    VARCHAR(50)
   external_id               VARCHAR(100) (nullable)
   url                       TEXT (nullable)
   title                     VARCHAR(200)
   company                   VARCHAR(200)
   location                  VARCHAR(200) (nullable)
   salary_min                INTEGER (nullable)
   salary_max                INTEGER (nullable)
   salary_raw                VARCHAR(100) (nullable)
   job_type                  VARCHAR(50) (nullable)
   description               TEXT (nullable)
   match_score               FLOAT (nullable)
   score_reason              TEXT (nullable)
   status                    VARCHAR(11)
   scraped_at                DATETIME
   updated_at                DATETIME

📋 resumes (11 columns)
   id                        INTEGER ← PK
   job_id                    INTEGER (nullable)
   is_base                   BOOLEAN
   version                   INTEGER
   target_role     

In [14]:
#insert seed data
from app.db.base import AsyncSessionLocal
from app.db.models import Job, JobStatus, Resume, Application, ApplyMethod, ApplicationStatus, Feedback
from datetime import datetime, timezone

async def seed():
    async with AsyncSessionLocal() as session:

        # 1. Base resume
        base_resume = Resume(
            is_base=True,
            version=1,
            target_role="ML Engineer",
            resume_text="""John Doe | ML Engineer
Skills: Python, PyTorch, LangChain, FastAPI, SQL
Experience: 3 years building ML pipelines and agentic AI systems.""",
            cover_letter=None,
        )
        session.add(base_resume)

        # 2. A scraped job
        job1 = Job(
            source="linkedin",
            external_id="li_123456",
            url="https://linkedin.com/jobs/view/123456",
            title="Senior ML Engineer",
            company="TechCorp",
            location="Hyderabad, India",
            salary_min=1800000,
            salary_max=2500000,
            salary_raw="₹18L–₹25L",
            job_type="full-time",
            description="Build and deploy ML models at scale using PyTorch and LangChain.",
            match_score=82.5,
            score_reason="Strong match on PyTorch, LangChain, and FastAPI. Missing Kubernetes experience.",
            status=JobStatus.shortlisted,
        )
        session.add(job1)

        # 3. A second job (lower score)
        job2 = Job(
            source="indeed",
            external_id="in_789012",
            url="https://indeed.com/viewjob?jk=789012",
            title="Data Scientist",
            company="DataCo",
            location="Bangalore, India",
            salary_raw="Competitive",
            job_type="remote",
            description="Statistical modelling and dashboard reporting.",
            match_score=54.0,
            score_reason="Partial match. Role focuses on BI/reporting, not ML engineering.",
            status=JobStatus.new,
        )
        session.add(job2)

        await session.flush()  # get IDs before linking

        # 4. Tailored resume for job1
        tailored_resume = Resume(
            job_id=job1.id,
            is_base=False,
            version=1,
            target_role="Senior ML Engineer",
            resume_text="""John Doe | Senior ML Engineer
Skills: Python, PyTorch, LangChain, FastAPI, SQL, Distributed Systems
Tailored for TechCorp — emphasised LangChain agent experience.""",
            cover_letter="Dear TechCorp team, I am excited to apply for the Senior ML Engineer role...",
            keywords_added="distributed systems, model serving, MLOps",
            changes_summary="Added distributed systems experience, reordered skills to match JD.",
        )
        session.add(tailored_resume)

        await session.flush()

        # 5. Application for job1
        app1 = Application(
            job_id=job1.id,
            resume_id=tailored_resume.id,
            apply_method=ApplyMethod.email,
            status=ApplicationStatus.sent,
            applied_to="careers@techcorp.com",
        )
        session.add(app1)

        # 6. Weekly feedback record
        fb = Feedback(
            week_number=1,
            year=2025,
            jobs_scraped=15,
            jobs_applied=3,
            rejections=1,
            interviews=0,
            avg_match_score=71.2,
            top_rejection_reasons='["Missing Kubernetes", "No cloud certifications"]',
            suggested_keywords="Kubernetes, AWS, MLOps",
            resume_improvements="Add a cloud deployment project to experience section.",
            resume_version_used=1,
        )
        session.add(fb)

        await session.commit()
        print("✅ Seed data inserted")
        print(f"   Base resume   : id={base_resume.id}")
        print(f"   Job 1         : id={job1.id} | score={job1.match_score}")
        print(f"   Job 2         : id={job2.id} | score={job2.match_score}")
        print(f"   Tailored CV   : id={tailored_resume.id}")
        print(f"   Application   : id={app1.id}")
        print(f"   Feedback      : id={fb.id}")

await seed()

✅ Seed data inserted
   Base resume   : id=1
   Job 1         : id=1 | score=82.5
   Job 2         : id=2 | score=54.0
   Tailored CV   : id=2
   Application   : id=1
   Feedback      : id=1


In [15]:
#query and validate data
from sqlalchemy import select

async def validate():
    async with AsyncSessionLocal() as session:

        # All jobs ordered by match score
        result = await session.execute(select(Job).order_by(Job.match_score.desc()))
        jobs = result.scalars().all()
        print("📋 Jobs by match score:")
        for j in jobs:
            print(f"   [{j.match_score:5.1f}] {j.title} @ {j.company} — {j.status}")

        # Shortlisted only
        result = await session.execute(
            select(Job).where(Job.status == JobStatus.shortlisted)
        )
        shortlisted = result.scalars().all()
        print(f"\n✅ Shortlisted jobs : {len(shortlisted)}")

        # Applications
        result = await session.execute(select(Application))
        apps = result.scalars().all()
        print(f"✅ Applications     : {len(apps)}")

        # Resumes
        result = await session.execute(select(Resume))
        resumes = result.scalars().all()
        print(f"✅ Resumes          : {len(resumes)} ({sum(1 for r in resumes if r.is_base)} base, {sum(1 for r in resumes if not r.is_base)} tailored)")

        # Feedback
        result = await session.execute(select(Feedback))
        feedbacks = result.scalars().all()
        print(f"✅ Feedback records : {len(feedbacks)}")
        for f in feedbacks:
            print(f"   Week {f.week_number}/{f.year} — applied={f.jobs_applied}, rejected={f.rejections}, avg_score={f.avg_match_score}")

await validate()

📋 Jobs by match score:
   [ 82.5] Senior ML Engineer @ TechCorp — JobStatus.shortlisted
   [ 54.0] Data Scientist @ DataCo — JobStatus.new

✅ Shortlisted jobs : 1
✅ Applications     : 1
✅ Resumes          : 2 (1 base, 1 tailored)
✅ Feedback records : 1
   Week 1/2025 — applied=3, rejected=1, avg_score=71.2


In [16]:
#PHASE 2 SUMMARY
print("=" * 50)
print("  Phase 2 — Database Schema Complete")
print("=" * 50)
print()
print("  Tables ready:")
print("  → jobs          (scrape results + scores + status)")
print("  → resumes       (base + tailored versions)")
print("  → applications  (apply log + outcomes)")
print("  → feedback      (weekly improvement loop)")
print()
print("  Next: Phase 3 — Job Scraper")
print("  → Selenium scraper for LinkedIn")
print("  → BeautifulSoup scraper for Indeed")
print("  → search_jobs LangChain tool")

  Phase 2 — Database Schema Complete

  Tables ready:
  → jobs          (scrape results + scores + status)
  → resumes       (base + tailored versions)
  → applications  (apply log + outcomes)
  → feedback      (weekly improvement loop)

  Next: Phase 3 — Job Scraper
  → Selenium scraper for LinkedIn
  → BeautifulSoup scraper for Indeed
  → search_jobs LangChain tool


In [22]:
#remove seed data to get original data
import sqlite3
DB_PATH = r"c:\AI_ML\Zenith_projects\AI_JobHuntAgent\job_hunt.db"
conn = sqlite3.connect(DB_PATH)
conn.execute("DELETE FROM feedback")
conn.execute("DELETE FROM applications")
conn.execute("DELETE FROM resumes")
conn.execute("DELETE FROM jobs")
conn.commit()
conn.close()
print("✅ DB cleared")

✅ DB cleared


# PHASE 3 - JOB SCRAPER

In [17]:
#setup
import sys, os, logging
sys.path.insert(0, os.path.abspath(".."))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)

from dotenv import load_dotenv
load_dotenv("../.env")

from config.settings import get_settings
settings = get_settings()

print("✅ Setup complete")
print(f"   LLM          : {settings.llm_provider} / {settings.ollama_model}")
print(f"   Min score    : {settings.agent_min_match_score}")
print(f"   Max jobs/run : {settings.agent_max_jobs_per_run}")
print(f"   Dry run      : {settings.agent_dry_run}")
print(f"   LinkedIn email set : {bool(settings.linkedin_email)}")

✅ Setup complete
   LLM          : ollama / llama3
   Min score    : 60
   Max jobs/run : 20
   Dry run      : True
   LinkedIn email set : False


In [18]:
#test ollama scrorer
from app.scoring.scorer import score_job

sample_resume = """
John Doe — ML Engineer
Skills: Python, PyTorch, LangChain, FastAPI, SQL, Docker, Git
Experience: 3 years building ML pipelines, LLM integrations, and REST APIs.
Education: B.Tech Computer Science
"""

sample_jd = """
Senior ML Engineer — TechCorp, Hyderabad
We are looking for an ML Engineer with:
- 3+ years experience with Python and ML frameworks (PyTorch/TensorFlow)
- Experience building LLM-powered applications (LangChain preferred)
- Strong backend skills (FastAPI or Flask)
- Familiarity with cloud platforms (AWS/GCP)
- Experience with containerisation (Docker, Kubernetes)
"""

print("Scoring sample job against sample resume...")
result = score_job(sample_resume, sample_jd)

print(f"\n✅ Score        : {result.score}/100")
print(f"   Reason       : {result.reason}")
print(f"   Matched      : {result.matched_skills}")
print(f"   Missing      : {result.missing_skills}")
print(f"   Recommend    : {result.recommendation}")

Scoring sample job against sample resume...


16:36:49  INFO     httpx — HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"



✅ Score        : 90.0/100
   Reason       : The candidate has most of the required skills and experience, with a strong background in ML engineering. The only gap is the lack of cloud platform experience.
   Matched      : ['Python', 'PyTorch', 'LangChain', 'FastAPI', 'Docker']
   Missing      : ['Cloud platforms (AWS/GCP)']
   Recommend    : apply


In [19]:
# Test with a weak match to verify low scoring works
weak_jd = """
Senior Accountant — FinCorp
Requirements: CA qualification, 5+ years in financial reporting,
expertise in SAP, Tally, and GST filing. CPA preferred.
"""

weak_result = score_job(sample_resume, weak_jd)
print(f"Weak match score : {weak_result.score}/100  (expected: below 40)")
print(f"Recommendation   : {weak_result.recommendation}  (expected: skip)")

assert weak_result.score < 50, f"Expected low score, got {weak_result.score}"
print("✅ Weak match correctly scored low")

16:37:08  INFO     httpx — HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Weak match score : 20.0/100  (expected: below 40)
Recommendation   : skip  (expected: skip)
✅ Weak match correctly scored low


In [20]:
#test linkedin scraper
if not settings.linkedin_email:
    print("⚠️  LINKEDIN_EMAIL not set in .env — skipping LinkedIn test")
    print("   Add credentials to .env to enable this test")
else:
    from app.scrapers.linkedin_scraper import scrape_linkedin

    print("Running LinkedIn scraper (max 3 jobs, slow by design)...")
    li_jobs = scrape_linkedin(
        query="ML Engineer",
        location="Hyderabad, India",
        max_jobs=3,
    )

    print(f"\n✅ LinkedIn: {len(li_jobs)} jobs scraped")
    for j in li_jobs:
        print(f"   [{j.external_id}] {j.title} @ {j.company} — {j.location}")
        print(f"     Description length: {len(j.description)} chars")

⚠️  LINKEDIN_EMAIL not set in .env — skipping LinkedIn test
   Add credentials to .env to enable this test


In [21]:
#test indeed scraper
from app.scrapers.indeed_scraper import scrape_indeed

print("Running Indeed scraper (max 3 jobs)...")
in_jobs = scrape_indeed(
    query="Machine Learning Engineer",
    location="Hyderabad",
    max_jobs=3,
)

print(f"\n✅ Indeed: {len(in_jobs)} jobs scraped")
for j in in_jobs:
    print(f"   [{j.external_id}] {j.title} @ {j.company} — {j.location}")
    print(f"     Salary: {j.salary_raw or 'not listed'}")
    print(f"     Description: {len(j.description)} chars")

16:38:58  INFO     app.scrapers.indeed_scraper — Indeed: fetching page (start=0) — https://in.indeed.com/jobs?q=Machine+Learning+Engineer&l=Hyderabad&sort=date&fromage=1&start=0


Running Indeed scraper (max 3 jobs)...


16:39:01  ERROR    app.scrapers.indeed_scraper — Indeed request failed: 403 Client Error: Forbidden for url: https://in.indeed.com/jobs?q=Machine+Learning+Engineer&l=Hyderabad&sort=date&fromage=1&start=0
16:39:01  INFO     app.scrapers.indeed_scraper — Indeed scrape complete: 0 jobs



✅ Indeed: 0 jobs scraped


In [22]:
#run full pipeline test
from app.tools.search_jobs import run_search_jobs

print("Running full search_jobs pipeline...")
print(f"  Query    : ML Engineer")
print(f"  Location : Hyderabad, India")
print(f"  Sources  : indeed only (LinkedIn optional)")
print()

summary = await run_search_jobs(
    query="ML Engineer",
    location="Hyderabad, India",
    max_per_source=5,
    sources=["indeed"],    # swap to ["linkedin", "indeed"] once creds confirmed
)

print("\n📊 Pipeline Summary")
print(f"   Scraped          : {summary['scraped']}")
print(f"   After dedup      : {summary['after_dedup']}")
print(f"   Above threshold  : {summary['above_threshold']} (min score {settings.agent_min_match_score})")
print(f"   Saved to DB      : {summary['saved_new']}")
print(f"   Skipped (dupes)  : {summary.get('skipped_duplicates', 0)}")

print("\n🏆 Top matches:")
for j in summary.get("top_jobs", []):
    print(f"   [{j['match_score']:5.1f}] {j['title']} @ {j['company']} — {j['recommendation']}")

16:39:22  INFO     app.scrapers.indeed_scraper — Indeed: fetching page (start=0) — https://in.indeed.com/jobs?q=ML+Engineer&l=Hyderabad%2C+India&sort=date&fromage=1&start=0


Running full search_jobs pipeline...
  Query    : ML Engineer
  Location : Hyderabad, India
  Sources  : indeed only (LinkedIn optional)



16:39:25  ERROR    app.scrapers.indeed_scraper — Indeed request failed: 403 Client Error: Forbidden for url: https://in.indeed.com/jobs?q=ML+Engineer&l=Hyderabad%2C+India&sort=date&fromage=1&start=0
16:39:25  INFO     app.scrapers.indeed_scraper — Indeed scrape complete: 0 jobs



📊 Pipeline Summary
   Scraped          : 0
   After dedup      : 0
   Above threshold  : 0 (min score 60)
   Saved to DB      : 0
   Skipped (dupes)  : 0

🏆 Top matches:


In [23]:
#save scraped jobs to DB
from sqlalchemy import select
from app.db.base import AsyncSessionLocal
from app.db.models import Job, JobStatus

async def check_db():
    async with AsyncSessionLocal() as session:
        result = await session.execute(
            select(Job).order_by(Job.match_score.desc())
        )
        all_jobs = result.scalars().all()

        shortlisted = [j for j in all_jobs if j.status == JobStatus.shortlisted]

        print(f"📋 Total jobs in DB  : {len(all_jobs)}")
        print(f"   Shortlisted       : {len(shortlisted)}")
        print()
        print("Top 10 by match score:")
        for j in all_jobs[:10]:
            score_str = f"{j.match_score:.1f}" if j.match_score else "N/A"
            print(f"   [{score_str:>5}] {j.title:<35} @ {j.company:<20} [{j.source}] {j.status}")
            if j.score_reason:
                print(f"           {j.score_reason[:80]}...")

await check_db()

📋 Total jobs in DB  : 2
   Shortlisted       : 1

Top 10 by match score:
   [ 82.5] Senior ML Engineer                  @ TechCorp             [linkedin] JobStatus.shortlisted
           Strong match on PyTorch, LangChain, and FastAPI. Missing Kubernetes experience....
   [ 54.0] Data Scientist                      @ DataCo               [indeed] JobStatus.new
           Partial match. Role focuses on BI/reporting, not ML engineering....


# Phase 4: Langchain ReAct Agent and tools

In [24]:
#setup
import sys, os, logging, json
sys.path.insert(0, os.path.abspath(".."))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)

from dotenv import load_dotenv
load_dotenv("../.env")

from config.settings import get_settings
settings = get_settings()

print("✅ Setup complete")
print(f"   Ollama model : {settings.ollama_model}")
print(f"   Min score    : {settings.agent_min_match_score}")
print(f"   Dry run      : {settings.agent_dry_run}")

✅ Setup complete
   Ollama model : llama3
   Min score    : 60
   Dry run      : True


In [25]:
#check DB state
from sqlalchemy import select
from app.db.base import AsyncSessionLocal
from app.db.models import Job, JobStatus, Resume

async def check_state():
    async with AsyncSessionLocal() as session:
        jobs = (await session.execute(
            select(Job).order_by(Job.match_score.desc().nullslast())
        )).scalars().all()

        resumes = (await session.execute(
            select(Resume).where(Resume.is_base == True)
        )).scalars().all()

        print(f"📋 Jobs in DB    : {len(jobs)}")
        print(f"   Base resumes  : {len(resumes)}")
        print()

        if not jobs:
            print("⚠️  No jobs found — run Phase 3 first to scrape jobs")
            return None
        if not resumes:
            print("⚠️  No base resume — run Phase 2 seed first")
            return None

        print("Top 5 jobs:")
        for j in jobs[:5]:
            score = f"{j.match_score:.1f}" if j.match_score else "unscored"
            print(f"   id={j.id:<3} [{score:>7}] {j.title:<35} @ {j.company}")

        return jobs[0].id   # return top job id for testing

top_job_id = await check_state()
print(f"\n→ Will use job_id={top_job_id} for tool tests")

📋 Jobs in DB    : 2
   Base resumes  : 1

Top 5 jobs:
   id=1   [   82.5] Senior ML Engineer                  @ TechCorp
   id=2   [   54.0] Data Scientist                      @ DataCo

→ Will use job_id=1 for tool tests


In [26]:
#test score job tool
from app.tools.score_job import run_score_job

if top_job_id:
    print(f"Scoring job_id={top_job_id}...")
    result = await run_score_job(top_job_id)

    print(f"\n✅ Score          : {result.get('score')}/100")
    print(f"   Recommendation : {result.get('recommendation')}")
    print(f"   Reason         : {result.get('reason')}")
    print(f"   Matched skills : {result.get('matched_skills')}")
    print(f"   Missing skills : {result.get('missing_skills')}")
else:
    print("⚠️  No jobs in DB — add jobs via Phase 3 first")

Scoring job_id=1...


16:42:04  INFO     httpx — HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
16:42:04  INFO     app.tools.score_job — score_job: job_id=1 'Senior ML Engineer' → 90.0/100



✅ Score          : 90.0/100
   Recommendation : apply
   Reason         : John has strong matching skills in PyTorch and LangChain, which are essential for the job. His experience building ML pipelines also aligns with the role's requirements.
   Matched skills : ['PyTorch', 'LangChain']
   Missing skills : []


In [27]:
#test pdf generator
from app.services.pdf_service import generate_pdf

sample_resume = """John Doe
john.doe@email.com | +91 99999 99999 | Hyderabad, India

SUMMARY
ML Engineer with 3 years experience building LLM-powered applications.

SKILLS
Python, PyTorch, LangChain, FastAPI, Docker, SQL, Git

EXPERIENCE
ML Engineer — TechStartup (2022–present)
- Built RAG pipeline serving 10k+ daily queries
- Reduced inference latency by 40% via model quantisation
- Led migration from monolith to FastAPI microservices

EDUCATION
B.Tech Computer Science — JNTU Hyderabad (2022)"""

pdf_bytes, pdf_path = generate_pdf(
    resume_text=sample_resume,
    job_title="Senior ML Engineer",
    company="TechCorp",
    version=1,
)

print(f"✅ PDF generated")
print(f"   Size     : {len(pdf_bytes):,} bytes")
print(f"   Saved to : {pdf_path}")

✅ PDF generated
   Size     : 2,269 bytes
   Saved to : outputs\resumes\resume_v1_Senior_ML_Engineer_TechCorp_20260411_164225.pdf


In [28]:
#rewrite resume
from app.tools.rewrite_resume import run_rewrite_resume

if top_job_id:
    print(f"Rewriting resume for job_id={top_job_id}...")
    print("(This calls Ollama 3 times — expect 30–90 seconds)")
    result = await run_rewrite_resume(top_job_id)

    if "error" in result:
        print(f"❌ Error: {result['error']}")
    else:
        print(f"\n✅ Resume rewritten")
        print(f"   Resume id      : {result['resume_id']}")
        print(f"   Version        : {result['version']}")
        print(f"   Keywords added : {result['keywords_added']}")
        print(f"   Changes        : {result['changes_summary']}")
        print(f"   PDF saved to   : {result['pdf_path']}")
        print(f"\n   Cover letter preview:")
        print(f"   {result['cover_letter_preview']}")
        tailored_resume_id = result['resume_id']
else:
    print("⚠️  No jobs in DB")

Rewriting resume for job_id=1...
(This calls Ollama 3 times — expect 30–90 seconds)


16:42:47  INFO     httpx — HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
16:43:08  INFO     httpx — HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
16:43:29  INFO     httpx — HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"



✅ Resume rewritten
   Resume id      : 3
   Version        : 2
   Keywords added : []
   Changes        : Added a summary section and reorganized the experience section to highlight specific accomplishments.
   PDF saved to   : outputs\resumes\resume_v2_Senior_ML_Engineer_TechCorp_20260411_164329.pdf

   Cover letter preview:
   Here is a concise, professional cover letter for the Senior ML Engineer role at TechCorp:

Dear Hiring Manager,

I'm excited about the opportunity to join TechCorp as a Senior ML Engineer, where I can...


In [32]:
from app.tools.apply_job import run_apply_job

# Find a job that hasn't been applied to yet
from sqlalchemy import select
from app.db.base import AsyncSessionLocal
from app.db.models import Job, JobStatus, Application

async def get_unapplied_job():
    async with AsyncSessionLocal() as session:
        # Get jobs that have NO application yet
        applied_job_ids = (
            await session.execute(select(Application.job_id))
        ).scalars().all()

        result = await session.execute(
            select(Job)
            .where(Job.id.notin_(applied_job_ids))
            .where(Job.match_score >= 60)
            .order_by(Job.match_score.desc())
            .limit(1)
        )
        job = result.scalars().first()
        if job:
            print(f"Found unapplied job: id={job.id} | {job.title} @ {job.company} | score={job.match_score}")
            return job.id
        else:
            print("⚠️  All jobs already have applications — adding a fresh job first")
            return None

unapplied_job_id = await get_unapplied_job()

⚠️  All jobs already have applications — adding a fresh job first


In [33]:
# job application tool test
if unapplied_job_id:
    print(f"Queuing application for job_id={unapplied_job_id}...")
    result = await run_apply_job(unapplied_job_id)

    if "error" in result:
        print(f"❌ Error: {result['error']}")
    elif "application_id" not in result:
        print(f"⚠️  Unexpected format: {result}")
    else:
        print(f"\n✅ Application queued")
        print(f"   Application id : {result['application_id']}")
        print(f"   Status         : {result['status']}")
        print(f"   Job            : {result['job_title']} @ {result['company']}")
        application_id = result['application_id']
else:
    print("⚠️  No unapplied jobs found — run Phase 3 scraper to get fresh jobs")

⚠️  No unapplied jobs found — run Phase 3 scraper to get fresh jobs


In [35]:
# Grab the existing application_id directly from DB
from sqlalchemy import select
from app.db.base import AsyncSessionLocal
from app.db.models import Application

async def get_existing_app_id():
    async with AsyncSessionLocal() as session:
        result = await session.execute(
            select(Application).order_by(Application.id.desc()).limit(1)
        )
        app = result.scalars().first()
        if app:
            print(f"Found application: id={app.id} | job_id={app.job_id} | status={app.status}")
            return app.id
        return None

application_id = await get_existing_app_id()
print(f"application_id set to: {application_id}")

Found application: id=1 | job_id=1 | status=ApplicationStatus.sent
application_id set to: 1


In [36]:
#test log results and feedback tool
from app.tools.log_result import run_log_result

# Log the application as rejected to test feedback loop
if 'application_id' in dir():
    print(f"Logging rejection for application_id={application_id}...")
    result = await run_log_result(
        application_id=application_id,
        outcome="rejected",
        notes="Automated rejection email received — missing Kubernetes experience",
    )

    print(f"\n✅ Result logged")
    print(f"   Outcome          : {result['outcome']}")
    print(f"   Job              : {result['job_title']} @ {result['company']}")

    su = result.get('strategy_update', {})
    if su.get('triggered'):
        print(f"\n🔁 Strategy update triggered!")
        print(f"   Top reasons    : {su.get('top_rejection_reasons')}")
        print(f"   Keywords       : {su.get('suggested_keywords')}")
        print(f"   Improvements   : {su.get('resume_improvements')}")
    else:
        print(f"\n   Strategy update : not triggered yet (need 3+ rejections)")
else:
    print("⚠️  No application_id — run cell 5 first")

Logging rejection for application_id=1...

✅ Result logged
   Outcome          : rejected
   Job              : Senior ML Engineer @ TechCorp

   Strategy update : not triggered yet (need 3+ rejections)


In [37]:
# Seed 2 more rejections to trigger the strategy update
from app.db.base import AsyncSessionLocal
from app.db.models import Application, ApplicationStatus, Job, ApplyMethod, Resume
from sqlalchemy import select

async def seed_extra_rejections():
    async with AsyncSessionLocal() as session:
        # Get any 2 other jobs to fake applications for
        result = await session.execute(
            select(Job).limit(3)
        )
        jobs = result.scalars().all()

        resume_result = await session.execute(
            select(Resume).where(Resume.is_base == True).limit(1)
        )
        resume = resume_result.scalars().first()

        created = 0
        for job in jobs[1:3]:   # skip first (already applied)
            app = Application(
                job_id=job.id,
                resume_id=resume.id,
                apply_method=ApplyMethod.manual,
                status=ApplicationStatus.rejected,
                applied_to=job.url,
                outcome_notes="Test rejection for strategy loop",
            )
            session.add(app)
            created += 1

        await session.commit()
        print(f"✅ Seeded {created} extra rejection(s)")

await seed_extra_rejections()

✅ Seeded 1 extra rejection(s)


In [38]:
# Now log another rejection — should trigger strategy update
if 'application_id' in dir():
    result = await run_log_result(
        application_id=application_id,
        outcome="rejected",
        notes="Second rejection — position filled internally",
    )

    su = result.get('strategy_update', {})
    if su.get('triggered'):
        print("🔁 Strategy update triggered!")
        print(f"   Rejection reasons : {su['top_rejection_reasons']}")
        print(f"   Suggested keywords: {su['suggested_keywords']}")
        print(f"   Resume advice     : {su['resume_improvements']}")
    else:
        print("Strategy update not triggered — check rejection count")

Strategy update not triggered — check rejection count


In [6]:
from langgraph.prebuilt import create_react_agent
from langchain_ollama import ChatOllama

print("✅ LangGraph ready")
print("✅ ChatOllama ready")

✅ LangGraph ready
✅ ChatOllama ready


In [2]:
#run full ReAct agent test
from app.agent.agent import agent, run_agent

print("Building ReAct agent...")
executor = agent(verbose=True)
print("✅ Agent ready")
print("   Tools: search_jobs, score_job, rewrite_resume, apply_job, log_result")

Building ReAct agent...
✅ Agent ready
   Tools: search_jobs, score_job, rewrite_resume, apply_job, log_result


In [3]:
# Run agent with a simple single-task instruction first
# (full hunt takes 5-10 min — start small)
print("Running agent: score all unscored jobs...")
print("=" * 60)

result = await run_agent(
    instruction="Score all jobs in the database that don't have a match score yet.",
    executor=executor,
)

print("=" * 60)
print(f"Status  : {result['status']}")
print(f"Steps   : {result['steps']}")
print(f"\nFinal answer:\n{result['output']}")

Running agent: score all unscored jobs...
Status  : ok
Steps   : 6

Final answer:
It seems like the job with ID 12345 does not exist in the database. Let me try again.

{"name": "score_job", "parameters": {"job_id":67890}}


In [4]:
# Full pipeline run — finds jobs, scores, rewrites, queues
# ⚠️  This will take 5–15 minutes depending on Ollama speed
print("Running full agent pipeline...")
print("(Expected: search → score → rewrite → queue applications)")
print("=" * 60)

full_result = await run_agent(
    instruction=(
        "Run a full job hunt cycle: "
        "1) Search for 'ML Engineer' jobs in Hyderabad on Indeed. "
        "2) Score each job against my resume. "
        "3) For any job scoring above 65, rewrite my resume. "
        "4) Queue applications for all tailored jobs. "
        "Give me a summary of what was done."
    ),
    executor=executor,
)

print("=" * 60)
print(f"Status : {full_result['status']}")
print(f"Steps  : {full_result['steps']}")
print(f"\nSummary:\n{full_result['output']}")

Running full agent pipeline...
(Expected: search → score → rewrite → queue applications)


LinkedIn credentials not set in .env — returning empty results
Indeed request failed: 403 Client Error: Forbidden for url: https://in.indeed.com/jobs?q=ML+Engineer&l=Hyderabad&sort=date&fromage=1&start=0


Status : ok
Steps  : 5

Summary:
{
  "job_id": 2,
  "title": "ML Engineer at ABC Inc.",
  "company": "ABC Inc.",
  "score": 85.0,
  "reason": "John's experience with TensorFlow and his ability to work with large datasets make him a strong candidate for this role.",
  "matched_skills": [
    "TensorFlow"
  ],
  "missing_skills": ["Keras"],
  "recommendation": "apply"
}


In [ ]:
#final db audit
import sys, os

# Point to the correct project root
PROJECT_ROOT = r"c:\AI_ML\Zenith_projects\AI_JobHuntAgent"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Path fixed:", sys.path[0])

# Now imports will work
from app.db.base import AsyncSessionLocal
from app.db.models import Job, Resume, Application, Feedback
from sqlalchemy import select

async def full_audit():
    async with AsyncSessionLocal() as session:
        jobs      = (await session.execute(select(Job))).scalars().all()
        apps      = (await session.execute(select(Application))).scalars().all()
        resumes   = (await session.execute(select(Resume))).scalars().all()
        feedbacks = (await session.execute(select(Feedback))).scalars().all()

        print("\n📊 Database Audit")
        print(f"   Jobs         : {len(jobs)}")
        scored      = [j for j in jobs if j.match_score]
        shortlisted = [j for j in jobs if j.status.value == 'shortlisted']
        applied     = [j for j in jobs if j.status.value == 'applied']
        print(f"     Scored     : {len(scored)}")
        print(f"     Shortlisted: {len(shortlisted)}")
        print(f"     Applied    : {len(applied)}")
        print(f"\n   Resumes      : {len(resumes)}")
        print(f"   Applications : {len(apps)}")
        print(f"   Feedback     : {len(feedbacks)}")

await full_audit()

Path fixed: c:\AI_ML\Zenith_projects\AI_JobHuntAgent

📊 Database Audit
   Jobs         : 2
     Scored     : 2
     Shortlisted: 0
     Applied    : 0

   Resumes      : 3
   Applications : 2
   Feedback     : 1


In [4]:
#Phase 4 summary
print("=" * 55)
print("  Phase 4 — LangChain ReAct Agent Complete")
print("=" * 55)
print()
print("  Files created:")
print("  → agent/tools/score_job.py")
print("  → agent/tools/rewrite_resume.py")
print("  → agent/tools/apply_job.py")
print("  → agent/tools/log_result.py")
print("  → agent/agent.py")
print("  → services/pdf_service.py")
print()
print("  Agent capabilities:")
print("  ✅ score_job       — Ollama match scoring saved to DB")
print("  ✅ rewrite_resume  — tailored text + cover letter + PDF")
print("  ✅ apply_job       — pending_approval queue (safe)")
print("  ✅ log_result      — outcome tracking + strategy loop")
print("  ✅ ReAct agent     — orchestrates full pipeline")
print()
print("  Next: Phase 5 — FastAPI Backend")
print("  → POST /agent/run — trigger agent run via API")
print("  → GET  /jobs      — list jobs with filters")
print("  → POST /resume/upload — upload base resume")
print("  → GET  /applications  — list pending approvals")
print("  → POST /applications/{id}/approve")
print("  → GET  /feedback  — strategy advice history")

  Phase 4 — LangChain ReAct Agent Complete

  Files created:
  → agent/tools/score_job.py
  → agent/tools/rewrite_resume.py
  → agent/tools/apply_job.py
  → agent/tools/log_result.py
  → agent/agent.py
  → services/pdf_service.py

  Agent capabilities:
  ✅ score_job       — Ollama match scoring saved to DB
  ✅ rewrite_resume  — tailored text + cover letter + PDF
  ✅ apply_job       — pending_approval queue (safe)
  ✅ log_result      — outcome tracking + strategy loop
  ✅ ReAct agent     — orchestrates full pipeline

  Next: Phase 5 — FastAPI Backend
  → POST /agent/run — trigger agent run via API
  → GET  /jobs      — list jobs with filters
  → POST /resume/upload — upload base resume
  → GET  /applications  — list pending approvals
  → POST /applications/{id}/approve
  → GET  /feedback  — strategy advice history


# Phase 5: FastAPI backend

In [6]:
#setup
import sys, os, httpx
sys.path.insert(0, os.path.abspath(".."))

BASE = "http://localhost:8000"

async def get(path, params=None):
    async with httpx.AsyncClient() as c:
        r = await c.get(f"{BASE}{path}", params=params, timeout=10)
        return r.status_code, r.json()

async def post(path, body=None):
    async with httpx.AsyncClient() as c:
        r = await c.post(f"{BASE}{path}", json=body, timeout=10)
        return r.status_code, r.json()

async def patch(path, body=None):
    async with httpx.AsyncClient() as c:
        r = await c.patch(f"{BASE}{path}", json=body, timeout=10)
        return r.status_code, r.json()

# Health check
status, data = await get("/health")
if status == 200:
    print(f"✅ FastAPI running — {data}")
else:
    print(f"❌ API not reachable. Start uvicorn first.")

✅ FastAPI running — {'status': 'ok', 'env': 'development'}


In [7]:
#test agent endpoint
# Trigger a dry-run agent cycle
status, data = await post("/api/agent/run", {
    "query": "ML Engineer",
    "location": "Hyderabad, India",
    "max_jobs": 3,
    "auto_apply": False,
    "dry_run": True
})
print(f"POST /api/agent/run → {status}")
print(f"  {data}")

POST /api/agent/run → 200
  {'status': 'started', 'message': "Agent started — searching for 'ML Engineer' in Hyderabad, India", 'jobs_scraped': None, 'jobs_applied': None}


In [8]:
# Check agent status
import asyncio
await asyncio.sleep(2)  # give it a moment

status, data = await get("/api/agent/status")
print(f"GET /api/agent/status → {status}")
print(f"  running   : {data['running']}")
print(f"  last_run  : {data['last_run']}")

GET /api/agent/status → 200
  running   : True
  last_run  : None


In [9]:
# List all jobs
status, jobs = await get("/api/jobs/")
print(f"GET /api/jobs/ → {status} | {len(jobs)} jobs")
for j in jobs[:3]:
    print(f"  [{j['match_score']:5.1f}] {j['title']} @ {j['company']} — {j['status']}")

GET /api/jobs/ → 200 | 2 jobs
  [ 90.0] Senior ML Engineer @ TechCorp — rejected
  [ 54.0] Data Scientist @ DataCo — new


In [10]:
# Dashboard stats
status, stats = await get("/api/jobs/stats")
print(f"GET /api/jobs/stats → {status}")
for k, v in stats.items():
    print(f"  {k:<20} : {v}")

GET /api/jobs/stats → 200
  total                : 2
  shortlisted          : 0
  applied              : 0
  interviews           : 0
  avg_match_score      : 72.0


In [11]:
# Filter — shortlisted only with score >= 70
status, jobs = await get("/api/jobs/", params={"status": "shortlisted", "min_score": 70})
print(f"GET /api/jobs/?status=shortlisted&min_score=70 → {status} | {len(jobs)} jobs")

GET /api/jobs/?status=shortlisted&min_score=70 → 200 | 0 jobs


In [12]:
# Update a job status manually
if jobs:
    job_id = jobs[0]['id']
    status, data = await patch(f"/api/jobs/{job_id}/status", {"status": "applied"})
    print(f"PATCH /api/jobs/{job_id}/status → {status} | {data}")

In [13]:
#test resume endpoint
# Upload a base resume
status, data = await post("/api/resumes/base", {
    "target_role": "ML Engineer",
    "resume_text": """Your Name | ML Engineer
Skills: Python, PyTorch, LangChain, FastAPI, PostgreSQL
Experience: 3 years building ML pipelines and agentic AI systems.
Education: B.Tech Computer Science"""
})
print(f"POST /api/resumes/base → {status}")
print(f"  id={data.get('id')} | version={data.get('version')} | is_base={data.get('is_base')}")

POST /api/resumes/base → 200
  id=4 | version=2 | is_base=True


In [14]:
# Get base resume
status, data = await get("/api/resumes/base")
print(f"GET /api/resumes/base → {status}")
print(f"  version={data.get('version')} | role={data.get('target_role')}")
print(f"  text preview: {data.get('resume_text', '')[:80]}...")

GET /api/resumes/base → 200
  version=2 | role=ML Engineer
  text preview: Your Name | ML Engineer
Skills: Python, PyTorch, LangChain, FastAPI, PostgreSQL
...


In [15]:
# Application funnel
status, funnel = await get("/api/applications/funnel")
print(f"GET /api/applications/funnel → {status}")
for stage, count in funnel.items():
    bar = '█' * count
    print(f"  {stage:<12} {bar} {count}")

GET /api/applications/funnel → 200
  pending_approval  0
  sent          0
  no_response   0
  rejected     ██ 2
  interview     0
  offer         0


In [16]:
# List all applications
status, apps = await get("/api/applications/")
print(f"GET /api/applications/ → {status} | {len(apps)} applications")
for a in apps:
    print(f"  id={a['id']} job_id={a['job_id']} status={a['status']} method={a['apply_method']}")

GET /api/applications/ → 200 | 2 applications
  id=2 job_id=2 status=rejected method=manual
  id=1 job_id=1 status=rejected method=email


In [17]:
#test feedback endpoint
# Latest feedback
status, fb = await get("/api/feedback/latest")
print(f"GET /api/feedback/latest → {status}")
if status == 200:
    print(f"  Week {fb['week_number']}/{fb['year']}")
    print(f"  scraped={fb['jobs_scraped']} applied={fb['jobs_applied']} rejected={fb['rejections']}")
    print(f"  avg_score={fb['avg_match_score']}")
    print(f"  improvements: {fb.get('resume_improvements', 'none')}")
else:
    print(f"  {fb['detail']}")

GET /api/feedback/latest → 200
  Week 1/2025
  scraped=15 applied=3 rejected=1
  avg_score=71.2
  improvements: Add a cloud deployment project to experience section.


In [18]:
#explore auto generated API docs
print("=" * 50)
print("  FastAPI auto-generated docs")
print("=" * 50)
print()
print("  Swagger UI  → http://localhost:8000/docs")
print("  ReDoc       → http://localhost:8000/redoc")
print("  OpenAPI JSON→ http://localhost:8000/openapi.json")
print()
print("  Open any of these in your browser to explore")
print("  and test all endpoints interactively.")

  FastAPI auto-generated docs

  Swagger UI  → http://localhost:8000/docs
  ReDoc       → http://localhost:8000/redoc
  OpenAPI JSON→ http://localhost:8000/openapi.json

  Open any of these in your browser to explore
  and test all endpoints interactively.


In [19]:
#phase 5 summary
print("=" * 50)
print("  Phase 5 — FastAPI Backend Complete")
print("=" * 50)
print()
print("  Endpoints ready:")
print("  → POST /api/agent/run       trigger agent cycle")
print("  → GET  /api/agent/status    check if running")
print("  → GET  /api/jobs/           list + filter jobs")
print("  → GET  /api/jobs/stats      dashboard header counts")
print("  → GET  /api/applications/funnel  pipeline chart data")
print("  → POST /api/resumes/base    upload master resume")
print("  → GET  /api/feedback/latest weekly improvement summary")
print()
print("  Next: Phase 6 — React Dashboard")
print("  → Jobs table with score + status filters")
print("  → Application funnel chart")
print("  → Weekly feedback panel")
print("  → Run agent button")

  Phase 5 — FastAPI Backend Complete

  Endpoints ready:
  → POST /api/agent/run       trigger agent cycle
  → GET  /api/agent/status    check if running
  → GET  /api/jobs/           list + filter jobs
  → GET  /api/jobs/stats      dashboard header counts
  → GET  /api/applications/funnel  pipeline chart data
  → POST /api/resumes/base    upload master resume
  → GET  /api/feedback/latest weekly improvement summary

  Next: Phase 6 — React Dashboard
  → Jobs table with score + status filters
  → Application funnel chart
  → Weekly feedback panel
  → Run agent button
